# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from ecohome.agent import Agent

In [2]:
# patch the event loop
import nest_asyncio
# allows asyncio.run() inside Jupyter's loop
nest_asyncio.apply()  

In [3]:
## Create the agent's instructions

ECOHOME_SYSTEM_PROMPT = '''
You are EcoHome Energy Advisor, an expert AI assistant specialized in optimizing residential energy systems (solar PV, EV charging, HVAC, appliances, battery storage). Your primary objective is to provide data-driven, personalized recommendations that minimize customer electricity cost and environmental impact while respecting user constraints.

Behavioral rules:
- Always prefer using the available tools for factual and numeric data. Do NOT fabricate numbers. When you need forecasts, historical data, prices, or savings calculations, call the appropriate tool(s) listed below.
- If required data is missing or ambiguous, ask 1-2 clear clarifying questions before giving final recommendations.
- Return both a concise human-readable recommendation and a machine-readable 'action_plan' JSON object (structure shown below) so results can be consumed programmatically.
- Be concise, explain key reasoning steps, and list assumptions and confidence (0.0–1.0).

Preferred tools (use these for authoritative values):
- get_weather_forecast(location, days): weather and estimated solar irradiance
- get_electricity_prices(date): hourly time-of-use prices and peak definitions
- query_energy_usage(start_date, end_date, device_type): historical consumption and costs
- query_solar_generation(start_date, end_date): historical solar production
- get_recent_energy_summary(hours): snapshot of recent usage & generation
- search_energy_tips(query): retrieval-augmented content for best practices
- calculate_energy_savings(device_type, current_usage_kwh, optimized_usage_kwh, price_per_kwh): savings estimates

Instructions for recommendations:
1) Gather data: call the minimal set of tools needed to answer the user's question (prefer tool calls over guessing).
2) Analyze trade-offs: provide at least two modes where applicable (cost_optimized, solar_optimized, balanced).
3) Provide schedule: for any device scheduling recommendation, provide timezone-aware ISO8601 start/end times and explain expected solar contribution and cost impact.
4) Quantify savings: show kWh, estimated cost, and estimated savings (use calculate_energy_savings when appropriate).
5) Transparency: list tools called (with their input parameters), show core calculation steps, and state any assumptions or missing data that would improve accuracy.

Output format (REQUIRED): Return a short human summary followed by a JSON-like structure exactly matching this example so downstream code can parse it:
{
  'summary': 'short text',
  'recommendation': 'short text',
  'action_plan': {
    'mode': 'solar_optimized|cost_optimized|balanced',
    'device': 'EV|HVAC|dryer|washer|other',
    'start': 'ISO8601 local start',
    'end': 'ISO8601 local end',
    'expected_solar_pct': 0-100,
    'estimated_cost_usd': numeric,
    'estimated_savings_usd': numeric
  },
  'tools_used': [
, 
] ,
  'assumptions': ['...'],
  'confidence': 0.0
}

Safety and quality:
- Do not hallucinate specific tariff or generation numbers — call tools instead. If a required API key or data is unavailable, say so explicitly and provide the best next steps and clarifying questions.
- Keep human explanations short (2-6 sentences) and ensure the 'action_plan' contains the numeric details needed for programmatic evaluation.

Be succinct and data-driven.'''

In [4]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [5]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [6]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power for charging your electric vehicle (EV) tomorrow in San Francisco, you should charge during the late morning to early afternoon when solar generation is expected to be higher and electricity rates are lower.

### Recommendations:
1. **Charging Window**: Charge your EV from **11:00 AM to 2:00 PM**.
2. **Expected Solar Contribution**: During this time, solar irradiance is estimated to be around 104 W/m² at 11:00 AM and will decrease to 56 W/m² by 2:00 PM, indicating good solar generation potential.
3. **Electricity Rates**: The cost is consistent at **$0.13 per kWh** throughout the day, as it is a weekend with no peak pricing.

### Action Plan:
- **Start Charging**: 2023-10-07T11:00:00-07:00
- **End Charging**: 2023-10-07T14:00:00-07:00
- **Expected Solar Contribution**: Approximately 50-70% of the charging needs can be met by solar generation during this window.

Here’s the structured action plan:

```json
{
  "summary": "Charge your EV from 11

In [7]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices
- get_weather_forecast


## 2. Define Test Cases

In [8]:
# Comprehensive test cases for the Energy Advisor (10 scenarios)
# Each test case includes: id, question, expected_tools, expected_response
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Time window recommendation, cost analysis, solar contribution estimate",
    },
    {
        "id": "hvac_setback_1",
        "question": "What's the most cost-effective thermostat schedule for my 3-bedroom house during a hot afternoon with lots of sun?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "query_energy_usage"],
        "expected_response": "Setback schedule (start/end), cooling temperature targets, estimated cost and comfort trade-off",
    },
    {
        "id": "appliance_schedule_1",
        "question": "When should I run the dishwasher and dryer tomorrow to minimize cost while maximizing rooftop solar usage?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "get_recent_energy_summary"],
        "expected_response": "Device start/end times, expected solar percent, cost comparison vs off-peak",
    },
    {
        "id": "solar_forecast_1",
        "question": "How much solar generation should I expect next 3 days and which day is best for heavy loads?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation"],
        "expected_response": "Daily generation estimates, recommended day/time for heavy loads, confidence/assumptions",
    },
    {
        "id": "cost_savings_1",
        "question": "If I shift my EV charging from 6pm to 2am every day, what's the estimated monthly savings?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings", "query_energy_usage"],
        "expected_response": "Monthly savings estimate (USD), kWh reduction, ROI note",
    },
    {
        "id": "battery_dispatch_1",
        "question": "When should I discharge my home battery tomorrow to minimize bills while preserving backup capacity (20% reserve)?",
        "expected_tools": ["get_electricity_prices", "get_recent_energy_summary"],
        "expected_response": "Dispatch schedule with state-of-charge constraints, expected cost impact and solar offset",
    },
    {
        "id": "demand_response_1",
        "question": "I received a demand response event notice for tomorrow 5–8pm. How should I reduce loads safely?",
        "expected_tools": ["get_electricity_prices", "get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "Actionable load reduction steps, prioritized devices, rough savings estimate and safety notes",
    },
    {
        "id": "thermostat_energy_tradeoff_1",
        "question": "Compare energy and cost impact of raising thermostat 2°C during peak hours vs using ceiling fans.",
        "expected_tools": ["query_energy_usage", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "kWh and cost comparison, recommended action, confidence and assumptions",
    },
    {
        "id": "ev_arrival_charge_1",
        "question": "My EV arrives at 18:30 and needs 20 kWh before 07:00. Provide a charging schedule that uses as much solar as possible.",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Time-window schedule with energy allocation, expected solar percent, cost estimate and any user prompts required",
    },
    {
        "id": "tips_search_1",
        "question": "Provide best practices to reduce standby energy for home electronics.",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "Short list of practical tips with citations from the knowledge base",
    },
]

# Ensure there are at least 10 test cases (the notebook checks this later)
if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: hvac_setback_1
Question: What's the most cost-effective thermostat schedule for my 3-bedroom house during a hot afternoon with lots of sun?
--------------------------------------------------

Test 3: appliance_schedule_1
Question: When should I run the dishwasher and dryer tomorrow to minimize cost while maximizing rooftop solar usage?
--------------------------------------------------

Test 4: solar_forecast_1
Question: How much solar generation should I expect next 3 days and which day is best for heavy loads?
--------------------------------------------------

Test 5: cost_savings_1
Question: If I shift my EV charging from 6pm to 2am every day, what's the estimated monthly savings?
--------------------------------------------------

Test 6: battery_dispatch_1
Question: When sh

## 4. Evaluate Responses

In [ ]:
from ecohome.utils import (
    build_ragas_metrics,
    generate_evaluation_report,
)

In [ ]:
# One-time setup
metrics = build_ragas_metrics(model="gpt-4o")

In [ ]:
# Run report
report = generate_evaluation_report(test_results, metrics=metrics)

In [ ]:
# Save
import json
with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)